[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-06-mt-evaluation.ipynb)


# CS 479 — Evaluating Machine Translation
### Lecture 6 in-class activity

By the end of this notebook you will have done, on a small scale, everything
Assignment 6 asks for on a large one: translate, score, and decide whether you
believe the score.

Work through it with the people near you. Nothing here is graded. The **Report Back**
at the bottom is what we will discuss as a class.

---


## Setup

`torchlingo` is the Matrix Lab's teaching library for neural MT. You will train models
with it later in the semester. Today we only want two functions out of it, which wrap
`sacrebleu` with defaults that are sane for a classroom.


In [ ]:
!pip install -q torchlingo sacrebleu


In [ ]:
from torchlingo.evaluation import compute_bleu
import sacrebleu

# chrF and TER come straight from sacrebleu, with all the references in one list.
def compute_chrf(preds, refs, word_order=2):
    return sacrebleu.corpus_chrf(preds, [refs], word_order=word_order)

def compute_ter(preds, refs):
    return sacrebleu.corpus_ter(preds, [refs])

print('sacrebleu', sacrebleu.__version__)


---
## Part 1 — The zero you have to understand

Two candidate translations. The first is *exactly* the reference. The second is close.
Predict the BLEU score before you run the cell.


In [ ]:
preds = ["Hello world", "How are you"]
refs  = ["Hello world", "How are you doing"]

bleu = compute_bleu(preds, refs)
print(bleu)
print('BLEU score:', round(bleu.score, 2))


### Why

Standard BLEU is the geometric mean of the 1-, 2-, 3- and 4-gram precisions,
multiplied by the brevity penalty:

$$\text{BLEU} = \text{BP} \cdot \exp\!\Big(\sum_{n=1}^{4} \lambda_n \log p_n\Big), \qquad \lambda_n = \tfrac14$$

A geometric mean is zero the moment **any** $p_n$ is zero. Neither of these sentences
is four words long, so there is no 4-gram to match anywhere in the corpus, $p_4 = 0$,
and $\log p_4 = -\infty$. The whole score collapses, no matter how good the translations are.

Look at the individual precisions in the printout above and confirm it for yourself.


In [ ]:
# The n-gram precisions behind that score
for n, p in enumerate(bleu.precisions, start=1):
    print(f'p{n} = {p:.2f}')


### The same two sentences, scored with chrF

chrF is an F-score over **character** n-grams (n = 1..6 by default), and it uses a
harmonic mean of precision and recall rather than a geometric mean of precisions only.
It degrades gracefully instead of collapsing.


In [ ]:
chrf = compute_chrf(preds, refs)
print('chrF score:', round(chrf.score, 2))


> **Talk about it.** Which of the two numbers better matches your own judgment of
> those two translations? Keep your answer; you will need it again at the end.


---
## Part 2 — A sample big enough to mean something

Below is a twenty-sentence English reference set and the output of two imaginary
MT systems. System A is fluent but drifts from the reference wording. System B is
literal and clumsy but sticks close to it.

Before you run anything: **which system would a human rank higher?** Write it down.


In [ ]:
references = [
    "The committee met on Tuesday to review the proposal.",
    "She walked to the market before the sun came up.",
    "The results were published in a leading medical journal.",
    "He said he would return the book by Friday afternoon.",
    "Heavy rain delayed the start of the football match.",
    "The museum opens at ten and closes at six on weekdays.",
    "They argued about the price for almost an hour.",
    "A small bridge crosses the river just north of town.",
    "The company announced record profits for the third quarter.",
    "My grandmother taught me how to make this soup.",
    "The train was cancelled without any explanation.",
    "Children under twelve may enter the park for free.",
    "We should finish the report before the meeting on Monday.",
    "The old church was destroyed by fire in 1892.",
    "She speaks four languages and is learning a fifth.",
    "The road to the village is closed during the winter.",
    "He forgot his passport and missed the flight.",
    "Local farmers depend on rain that no longer arrives.",
    "The film was praised by critics but ignored by audiences.",
    "Please turn off the lights when you leave the room.",
]

system_a = [  # fluent, freely worded
    "On Tuesday the committee gathered to look over the proposal.",
    "Before sunrise she made her way to the market on foot.",
    "A top medical journal carried the findings.",
    "He promised to bring the book back on Friday afternoon.",
    "The football match kicked off late because of heavy rain.",
    "On weekdays the museum is open from ten until six.",
    "For nearly an hour they quarrelled over the price.",
    "Just north of town, a little bridge spans the river.",
    "Record third-quarter profits were announced by the firm.",
    "This soup is my grandmother's recipe, taught to me by her.",
    "No reason was given for cancelling the train.",
    "Entry to the park is free for children under twelve.",
    "Let us wrap up the report ahead of Monday's meeting.",
    "Fire destroyed the old church in 1892.",
    "With four languages already, she is picking up a fifth.",
    "In winter the road leading to the village is shut.",
    "Having left his passport behind, he missed the flight.",
    "Rain that no longer comes is what local farmers rely on.",
    "Critics praised the film; audiences paid it no attention.",
    "Switch the lights off on your way out of the room.",
]

system_b = [  # literal, clumsy, close to the reference wording
    "The committee met on Tuesday to review the proposal document.",
    "She walked to the market before the sun came out.",
    "The results were published in a leading medical magazine.",
    "He said he will return the book by Friday afternoon.",
    "Heavy rain delayed the start of the football game.",
    "The museum opens at ten and closes at six in weekdays.",
    "They argued about the price for almost one hour.",
    "A small bridge crosses the river just north of the town.",
    "The company announced record profits for the third quarter year.",
    "My grandmother taught me how to do this soup.",
    "The train was cancelled without any explanations.",
    "Children under twelve can enter the park for free.",
    "We should finish the report before the meeting in Monday.",
    "The old church was destroyed by the fire in 1892.",
    "She speaks four languages and is learning the fifth.",
    "The road to the village is closed during winter.",
    "He forgot his passport and lost the flight.",
    "Local farmers depend on rain which no longer arrives.",
    "The film was praised by critics but ignored by the audiences.",
    "Please turn off the lights when you leave from the room.",
]

print(len(references), len(system_a), len(system_b))


In [ ]:
for name, out in (("System A (fluent, free)", system_a), ("System B (literal, close)", system_b)):
    b = compute_bleu(out, references)
    c = compute_chrf(out, references)
    print(f'{name:28s}  BLEU {b.score:6.2f}   chrF {c.score:6.2f}')


> **Talk about it.** Did the metrics pick the system you picked? If they did not,
> say precisely what BLEU is rewarding that you were not.

This is the whole of Assignment 6 in miniature, and it is why the assignment makes you
rank by hand **before** you look at any score.


---
## Part 3 — The brevity penalty, on purpose

BLEU is a precision-oriented metric. A system that emits only the words it is sure about
would score beautifully without a correction. The brevity penalty is that correction:

$$\text{BP} = \begin{cases} 1 & c > r \\ e^{(1 - r/c)} & c \le r \end{cases}$$

where $c$ is the candidate length and $r$ the reference length. Truncate every output to
its first few words and watch what the two metrics do.


In [ ]:
def truncate(sentences, k):
    return [' '.join(s.split()[:k]) for s in sentences]

print(f'{"words kept":>11} {"BLEU":>8} {"BP":>8} {"chrF":>8}')
for k in (3, 5, 8, 12, 100):
    out = truncate(system_b, k)
    b = compute_bleu(out, references)
    c = compute_chrf(out, references)
    print(f'{k:>11} {b.score:8.2f} {b.bp:8.3f} {c.score:8.2f}')


> **Talk about it.** At three words kept, the surviving words are almost all correct.
> What stops BLEU from rewarding that? Read the BP column.


---
## Part 4 — Your own data

Now the part that is genuinely the start of Assignment 6.

Upload the two cleaned, sentence-aligned files you submitted for Assignment 5 (use the
file pane on the left in Colab, or the cell below), then fill in the TODOs.


In [ ]:
from google.colab import files
uploaded = files.upload()   # pick your two files
print(list(uploaded))


In [ ]:
# TODO: point these at your two uploaded filenames.
SRC_FILE = ''   # your source-language file (English, if you translate out of English)
TGT_FILE = ''   # your target-language file, line-for-line aligned with SRC_FILE

with open(SRC_FILE, encoding='utf-8') as f:
    src_lines = [l.rstrip('\n') for l in f]
with open(TGT_FILE, encoding='utf-8') as f:
    tgt_lines = [l.rstrip('\n') for l in f]

assert len(src_lines) == len(tgt_lines), (len(src_lines), len(tgt_lines))
print(f'{len(src_lines):,} aligned pairs')


In [ ]:
# Take a random sample. Ten is enough for class; the assignment wants at least 500.
import random
random.seed(479)

N = 10
idx = random.sample(range(len(src_lines)), N)
sample_src = [src_lines[i] for i in idx]
sample_ref = [tgt_lines[i] for i in idx]

for s in sample_src:
    print(s)


### Translate them

Copy the ten source sentences printed above into an MT system in another tab
(translate.google.com, bing.com/translator, deepl.com, translate.yandex.com), then
paste the ten translations back below, **in the same order**.

For the assignment you will automate this for 500 or 1000 sentences. For the next ten
minutes, copy and paste is faster than an API key.


In [ ]:
# TODO: paste the machine translations here, one per line, same order as above.
hypotheses = """

""".strip().split('\n')

assert len(hypotheses) == N, f'got {len(hypotheses)}, expected {N}'


In [ ]:
# TODO: score your sample against your own target sentences as references.
b = compute_bleu(hypotheses, sample_ref)
c = compute_chrf(hypotheses, sample_ref)
t = compute_ter(hypotheses, sample_ref)

print('BLEU', round(b.score, 2))
print('chrF', round(c.score, 2))
print('TER ', round(t.score, 2), '(lower is better)')


In [ ]:
# Now rank them yourself, before you look at the numbers again.
# Print each source, your reference, and the MT output side by side.
for s, r, h in zip(sample_src, sample_ref, hypotheses):
    print('SRC:', s)
    print('REF:', r)
    print('MT :', h)
    print('-' * 70)


### If your language is not written with spaces

Chinese, Japanese, Thai, Khmer, Lao, Burmese: word-level BLEU is close to meaningless
for you, because the tokenizer has to guess where the words are. `compute_bleu` detects
this and switches to character tokenization, but chrF is the metric to trust. Say so in
your Assignment 6 analysis; it is exactly the kind of observation the analysis is for.


---
## Report Back

Be ready to give the class:

1. **Two numbers**: the BLEU and the chrF you got on your ten pairs.
2. **One sentence**: whether you agree with the ranking those numbers imply, having
   read the ten translations yourself.
3. **One count**: if your BLEU came out at or near zero, how many sentences did it take
   before the number started to move? (Rerun Part 4 with a larger `N` if you have time.)

A BLEU of zero on ten short sentences is the expected answer, not a bug. What matters is
that you can say why.

---

### Where this goes next

Assignment 6 is this notebook at scale, plus the piece no library can do for you: a human
ranking of 50 sentences in **MTEval**, done *before* you see any score, and a written
analysis of where the metrics and your own judgment parted company.
